# చాప్టర్ 7: చాట్ అప్లికేషన్లు నిర్మించడం  
## మైక్రోసాఫ్ట్ ఫౌండ్రి మోడల్స్ API త్రుటి ప్రారంభం  

ఈ నోటుబుక్ [Azure OpenAI సాంపిల్స్ రిపాజిటరీ](https://github.com/Azure/azure-openai-samples?WT.mc_id=academic-105485-koreyst) నుంచి అనుకూలీకరించబడింది, ఇది [Azure OpenAI](notebook-azure-openai.ipynb) సర్వీసులను యాక్సెస్ చేసే నోటుబుక్స్ కలిగి ఉంది.  

> **గమనిక:** GitHub మోడల్స్ జూలై 2026 చివరికి నివృత్తి చెందనున్నాయి. ఈ నోటుబుక్ ఇప్పుడు [Microsoft Foundry Models](https://ai.azure.com/catalog/models?WT.mc_id=academic-105485-koreyst) లక్ష్యంగా తీసుకుంటోంది, ఇది అదే ఉచిత ప్రయోగం అందించే మోడల్ కాటలాగ్ మరియు Azure AI ఇన్ఫరెన్స్ SDK అనుభవాన్ని అందిస్తుంది.  


# అవలోకనం  
"పెద్ద భాషా నమూనాలు టెక్స్ట్‌ను టెక్స్ట్‌కు మ్యాప్ చేసే ఫంక్షన్లు. ఇన్‌పుట్ టెక్స్ట్ స్ట్రింగ్ ఇచ్చినప్పుడు, పెద్ద భాషా నమూనా తర్వాత వచ్చే టెక్స్ట్‌ను అంచనా వేయడానికి ప్రయత్నిస్తుంది"(1). ఈ "క్విక్‌స్టార్ట్" నోట్‌బుక్ వాడుకరులకు హై-లెవల్ LLM కాన్సెప్ట్‌లు, AML ప్రారంభించడానికి cores_PACKAGE అవసరాలు, ప్రాంప్ట్ డిజైన్‌కు సాఫ్ట్ పరిచయం మరియు వివిధ ఉపయోగాల కొన్ని సులభ ఉదాహరణలను పరిచయం చేస్తుంది.  


## సూచన పట్టిక  

[అవలోకనం](#overview)  
[OpenAI సేవ ఎలా ఉపయోగించాలి](#how-to-use-openai-service)  
[1. మీ OpenAI సేవ సృష్టించడం](#1.-creating-your-openai-service)  
[2. సంస్థాపన](#2.-installation)    
[3. క్రెడెన్‌షియల్స్](#3.-credentials)  

[వినియోగ సందర్భాలు](#use-cases)    
[1. టెక్స్ట్ సంక్షిప్తం చేయాలై](#1.-summarize-text)  
[2. టెక్స్ట్ వర్గీకరణ](#2.-classify-text)  
[3. కొత్త ఉత్పత్తి పేర్లు సృష్టించండి](#3.-generate-new-product-names)  
[4. ఒక వర్గీకర్తను సరిచూడండి](#4.fine-tune-a-classifier)  

[సూచనలు](#references)


### మీ మొదటి ప్రాంప్ట్‌ని నిర్మించండి  
ఈ చిన్న వ్యాయామం మైక్రోసాఫ్ట్ ఫౌండ్రీ మోడల్స్‌లో ఒక సాధారణ పని "సారాంశం" కోసం ప్రాంప్ట్‌లు సమర్పించడానికిగురించి ప్రాథమిక పరిచయాన్ని అందిస్తుంది.  


**దశలు**:  
1. మీరు ఇంకా చేయకపోతే, మీ పایتాన్ ఎన్విరాన్‌మెంట్‌లో `azure-ai-inference` లైబ్రరీని ఇన్‌స్టాల్ చేయండి.  
2. స్టాండర్డ్ సహాయ లైబ్రరీలను లోడ్ చేసి, మీ మైక్రోసాఫ్ట్ ఫౌండ్రీ మోడల్స్ క్రెడెంషియల్స్ సెట్ చేయండి.  
3. మీ పనికి ఒక మోడల్‌ని ఎంచుకోండి  
4. మోడల్ కోసం ఒక సాదారణ ప్రాంప్ట్‌ని సృష్టించండి  
5. మీ అభ్యర్థనను మోడల్ API కి సమర్పించండి!  


### 1. `azure-ai-inference`ను ఇన్‌స్టాల్ చేయండి


In [ ]:
%pip install azure-ai-inference

### 2. సహాయక గ్రంథాలయాలు దిగుమతి చేయండి మరియు ప్రమాణపత్రాలను సృష్టించండి


In [ ]:
import os
from azure.ai.inference import ChatCompletionsClient
from azure.ai.inference.models import SystemMessage, UserMessage
from azure.core.credentials import AzureKeyCredential

# Get these from your Microsoft Foundry project's "Overview" page
token = os.environ["AZURE_INFERENCE_CREDENTIAL"]
endpoint = os.environ["AZURE_INFERENCE_ENDPOINT"]

client = ChatCompletionsClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(token),
)


### 3. సరైన మోడల్‌ను కనుగొనడం  
GPT-4o మరియు GPT-4o మినీ వంటి మోడల్స్ సహజ భాషను అర్థం చేసుకోగలవు మరియు సృష్టించగలవు, మరియు అవి Microsoft Foundry Models కాటలాగ్‌లో Meta, Mistral, Cohere, మరియు Microsoft మోడల్స్‌తో పాటు అందుబాటులో ఉన్నాయి.


In [ ]:
# Select a general purpose chat model
model_name = "gpt-5-mini"


## 4. ప్రాంప్ట్ డిజైన్  

"పెద్ద భాషా మోడల్స్ యొక్క మాయాజాలం ఏమిటంటే, విస్తృత పరిమాణంలో ఉన్న పాఠ్యంపై ఈ అంచనా దోషాన్ని తగ్గించేలా శిక్షణ పొందటం వలన, మోడల్స్ ఈ అంచనాలకు ఉపయోగకరంగా ఉండే భావనలు నేర్చుకుంటాయి. ఉదాహరణకు, అవి ఈ క్రింది భావనలను నేర్చుకుంటాయి"(1):

* ఎలా స్పెల్ చేయాలి
* వ్యాకరణం ఎలా పని చేస్తుంది
* ఎలా పునఃవాక్యం (paraphrase) చేయాలి
* ఎలా ప్రశ్నలకు సమాధానం ఇవ్వాలి
* ఎలా సంభాషణ నిలబెట్టుకోవాలి
* ఎలా అనేక భాషల్లో రాయాలి
* కోడ్ ఎలా రాయాలి
* ఇతరాలు.

#### పెద్ద భాషా మోడల్ ని ఎలా నియంత్రించాలి  
"పెద్ద భాషా మోడల్ కి అందుకునే అన్ని ఇన్పుట్లలో, అత్యంత ప్రభావశీలమైనది టెక్స్ట్ ప్రాంప్ట్(1)ే.

పెద్ద భాషా మోడల్స్ ని కొన్ని విధాలుగా ప్రాంప్ట్ చేయవచ్చు:

సూచన: మీరు కావాలనే మోడల్ కి చెప్పండి
పూర్తి చెల్లింపు: మీరు కావలసిన ప్రారంభ భాగాన్ని మోడల్ పూర్తి చేయించండి
ప్రదర్శన: మోడల్ కి మీరు కావలసినది చూపించండి, కింద ఇచ్చిన వాటితో ఏదైనా:
ప్రాంప్ట్ లో కొన్ని ఉదాహరణలు
ఫైన్-ట్యూనింగ్ శిక్షణ డేటాసెట్‌లో వందల లేదా వేల ఉదాహరణలు"



#### ప్రాంప్ట్ సృష్టించడానికి మూడు ప్రాథమిక మార్గదర్శకాలు ఉన్నాయి:

**చూపించి చెప్పండి**. మీరు ఏమి కోరుకుంటున్నారో స్పష్టంగా తెలియజేయండి, అది సూచనలు, ఉదాహరణలు లేదా వాటి మిశ్రమం ద్వారా కావచ్చు. మీరు మోడల్ ని ఒక అంశాల జాబితాను వర్ణమాల ఆర్డర్ లో క్రమబద్దీకరించమని కోరుకుంటే లేదా ఒక పేరాను భావ ప్రకారం వర్గీకరించమని కోరుకుంటే, మీరు అనుకుంటున్నదే చూపించండి.

**గుణాత్మక డేటా అందించండి**. మీరు ఒక వర్గీకర్త(క్లాసిఫైయర్) ని రూపొందించడానికి లేదా మోడల్ ని ఒక నమూనాను అనుసరించమని కోరుకుంటే, పరోక్షంగా సరిపడ మోతాదులో ఉదాహరణలు ఉండాలని చూడండి. మీ ఉదాహరణలను సిద్ధంగా సరిచూసుకోండి — మోడల్ సాధారణంగా ప్రాథమిక స్పెల్లింగ్ తప్పులను కనిపెట్టేంత తెలివైనది, కాని ఇది ఉద్దేశపూర్వకమని భావించి సమాధానం ప్రభావితం అయ్యే అవకాశం ఉంది.

**మీ సెట్టింగ్స్ ని పరిశీలించండి.** టెంపరేచర్ మరియు టాప్_పీ సెట్టింగ్స్ ఒకటే సరిగ్గా ఉన్న సమాధానాన్ని ఇవ్వడంలో మోడల్ ఎంత నిర్ణయాత్మకంగా ఉంటుంది అనేదాన్ని నియంత్రిస్తాయి. మీరు ఒకటే నిజమైన సమాధానం ఉన్న ప్రశ్న అడుగుతున్నట్లయితే, వీటిని తక్కువగా సెట్ చేయాలి. మీరు ఎక్కువ వైవిధ్యమైన సమాధానాలు కోరుతుంటే, వీటిని ఎక్కువగా సెట్ చేయవచ్చు. ఈ సెట్టింగ్స్ తో చాలా మంది తప్పుగా అనుకునేదేమిటంటే, ఇవి "తెలివితక్కువ" లేదా "సృష్టించటం" నియంత్రణలు అనుకుంటారు.


మూలం: https://learn.microsoft.com/azure/ai-foundry/openai/overview


### 5. సమర్పించండి!


In [ ]:
# Create your first prompt
text_prompt = "Should oxford commas always be used?"

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},])

response.choices[0].message.content

### అదే కాల్‌ను మళ్లీ చేయడం, ఫలితాలు ఎలా పోలిక పడతాయి?


In [ ]:

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},])

response.choices[0].message.content

## సారాంశం చేయండి  
#### సవాలు  
ఒక టెక్స్ట్ పాసేజ్ చివర 'tl;dr:' అని చేర్చడం ద్వారా టెక్స్ట్‌ను సారాంశం చేయండి. అదనపు సూచనలు లేకుండా ఒక పలు పనులు ఎలా చేయాలో మోడల్ ఎలా అర్థం చేసుకోవడం చూసి గమనించండి. మీరు tl;dr కంటే మరింత వివరణాత్మక ప్రాంప్ట్లతో ప్రయోగం చేయవచ్చు, తద్వారా మోడల్ ప్రవర్తనను మార్చి మీరు పొందే సారాంశాన్ని అనుకూలీకరించవచ్చు(3).  

ఇటీవలైన కార్యాలు పెద్ద పాఠ్య నిబంధనపై ప్రీ-ట్రైనింగ్ చేసి, తర్వాత ఒక నిర్దిష్ట పనిపై ఫైన్-ట్యూనింగ్ చేయడం ద్వారా అనేక NLP పనులు మరియు బెంచ్‌మార్క్‌లపై గణనీయమైన అభివృద్ధులు చూపాయి. సాధారణంగా ఆర్కిటెక్చర్‌లో టాస్క్-అగ్నోస్టిక్ అయినప్పటికీ, ఈ విధానం ఇంకా వేల లేదా పదిలక్షల ఉదాహరణలతో కూడిన టాస్క్-ప్రత్యేక ఫైన్-ట్యూనింగ్ డేటాసెట్‌లను అవసరపడుతుంది. తులనాత్మకంగా, మనుషులు సాధారణంగా కొద్ది ఉదాహరణల నుంచి లేదా సులభమైన సూచనల ద్వారా కొత్త భాషా పనిని చేయగలరు - ఇది ప్రస్తుత NLP వ్యవస్థలు ఇంకా ఎక్కువగా సవాలు ఎదుర్కొంటున్న విషయం. ఇక్కడ మేము చూపిస్తున్నాము భాషా మోడల్స్ పెంచడం ద్వారా టాస్క్-అగ్నోస్టిక్, ఫ్యూ-షాట్ పనితీరు గణనీయంగా మెరుగవుతుంది, అప్పుడప్పుడు మునుపటి అత్యుత్తమ ఫైన్-ట్యూనింగ్ పద్ధతులతో ప్రాయసిగా సమకూరుతుంది.  



tl;dr  


# అనేక వాడుక సందర్భాల కోసం వ్యాయామాలు  
1. టెక్స్ట్ సారాంశం చేయండి  
2. టెక్స్ట్ వర్గీకరణ  
3. కొత్త ఉత్పత్తి పేర్లు సృష్టించండి


In [ ]:
prompt = "Recent work has demonstrated substantial gains on many NLP tasks and benchmarks by pre-training on a large corpus of text followed by fine-tuning on a specific task. While typically task-agnostic in architecture, this method still requires task-specific fine-tuning datasets of thousands or tens of thousands of examples. By contrast, humans can generally perform a new language task from only a few examples or from simple instructions - something that current NLP systems still largely struggle to do. Here we show that scaling up language models greatly improves task-agnostic, few-shot performance, sometimes even reaching competitiveness with prior state-of-the-art fine-tuning approaches.\n\nTl;dr"


In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},])

response.choices[0].message.content

## వచనం వర్గీకరించండి  
#### సవాలు  
అనుమాన సమయంలో అందించబడిన వర్గాలలో వస్తువులను వర్గీకరించండి. దిగువ ఉదాహరణలో, మేము వర్గాలు మరియు వర్గీకరించవలసిన వచనాన్ని రెండింటిని కూడా ప్రాంప్ట్(*playground_reference) లో అందిస్తున్నాము. 

కస్టమర్ విచారణ: హలో, నా ల్యాప్‌టాప్ కీబోర్డ్‌లో ఒక కీ ఇటీవల పగిలిపోయింది మరియు నాకు ఒక ప్రత్యామ్నాయం అవసరం:

వర్గీకరించబడిన వర్గం:


In [ ]:
prompt = "Classify the following inquiry into one of the following: categories: [Pricing, Hardware Support, Software Support]\n\ninquiry: Hello, one of the keys on my laptop keyboard broke recently and I'll need a replacement:\n\nClassified category:"
print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},])

response.choices[0].message.content

## కొత్త ఉత్పత్తి పేర్లను తయారు చేయండి
#### సవాలు
ఉదాహరణ పదాల నుండి ఉత్పత్తి పేర్లను సృష్టించండి. ఇక్కడ మేము పేర్లను తయారు చేసుకునే ఉత్పత్తి గురించి సమాచారం ప్రాంప్ట్‌లో చేర్చాము. మేము కోరుకున్న నమూనాను చూపించడానికి ఒక సమాన ఉదాహరణను కూడా అందించాము. యాదృచ్చికత్వం మరియు మరింత సృజనాత్మకమైన ప్రతిస్పందనలను పెంచేందుకు టెంపరచర్ విలువ ను కూడా ఎక్కువగా పెట్టాము.

ఉత్పత్తి వివరణ: ఒక హోమ్ మిల్క్‌షేక్ తయారీ యంత్రం
బీజ పదాలు: వేగవంతమైన, ఆరోగ్యకరమైన, సన్నగా ఉండే.
ఉత్పత్తి పేర్లు: హోమ్‌షేకర్, ఫిట్ షేకర్, క్విక్‌షేక్, షేక్ మేకర్

ఉత్పత్తి వివరణ: ఏదేని కాలి పరిమాణంతో సరిపోయే షూల జంట.
బీజ పదాలు: అనుకూలించగల, సరిపడే, ఒమ్ని-ఫిట్.


In [ ]:
prompt = "Product description: A home milkshake maker\nSeed words: fast, healthy, compact.\nProduct names: HomeShaker, Fit Shaker, QuickShake, Shake Maker\n\nProduct description: A pair of shoes that can fit any foot size.\nSeed words: adaptable, fit, omni-fit."

print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt}])

response.choices[0].message.content

# సూచనలు  
- [Openai వంటకాల పుస్తకం](https://github.com/openai/openai-cookbook?WT.mc_id=academic-105485-koreyst)  
- [Microsoft Foundry పోర్టల్](https://ai.azure.com?WT.mc_id=academic-105485-koreyst)  
- [లేఖను వర్గీకరించడానికి GPT మోడల్స్‌ను ఫైన్-ట్యూన్ చేసే ఉత్తమ ఆచారాలు](https://platform.openai.com/docs/guides/fine-tuning?WT.mc_id=academic-105485-koreyst)


# మరింత సహాయం కోసం  
[OpenAI వాణిజ్యీకరణ బృందం](AzureOpenAITeam@microsoft.com) 


# బాధ్యత వహించే వారు
* [Chew-Yean Yam](https://www.linkedin.com/in/cyyam/)


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**అస్వీకరణ**:
ఈ పత్రం AI అనువాద సేవ [Co-op Translator](https://github.com/Azure/co-op-translator) ఉపయోగించి అనువదించబడింది. మేము ఖచ్చితత్వానికి ప్రయత్నిస్తున్నప్పటికీ, ఆటోమేటెడ్ అనువాదాలు తప్పులు లేదా అసమగ్రతలను కలిగి ఉండవచ్చు. దాని స్వదేశ భాషలో ఉన్న అసలు పత్రాన్ని అధికారం కలిగిన మూలంగా పరిగణించాలి. కీలకమైన సమాచారం కోసం, ప్రొఫెషనల్ మానవ అనువాదాన్ని సిఫారసు చేస్తాము. ఈ అనువాదం ఉపయోగం వల్ల కలిగే ఏవైనా అపార్థాలు లేదా తప్పుదారులు కోసం మేము బాధ్యత వహించము.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
